# Terraform — First Contact

IaC means defining infrastructure in code instead of clicking in consoles. It gives repeatability, version control, and speed.

Terraform uses a declarative model: you describe desired state, Terraform figures out actions.

Citi example: instead of waiting weeks for infra, a Data Engineer provisions S3, IAM, KMS in minutes.

[main.tf] → [plan] → [diff] → [apply] → [AWS resources]

In [1]:
import subprocess, os

os.environ["AWS_PROFILE"] = "study"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

tf_out = subprocess.run(["terraform", "version"], capture_output=True, text=True, encoding="utf-8", errors="replace")
aws_out = subprocess.run(["aws", "sts", "get-caller-identity"], capture_output=True, text=True, encoding="utf-8", errors="replace")

if tf_out.returncode != 0:
    raise RuntimeError(f"Terraform not found. Install from https://releases.hashicorp.com/terraform/\n{tf_out.stderr}")
print(tf_out.stdout.strip())

if aws_out.returncode != 0:
    raise RuntimeError(f"AWS CLI error: {aws_out.stderr}")
print(aws_out.stdout.strip())

Terraform v1.10.5
on windows_amd64

Your version of Terraform is out of date! The latest version
is 1.14.8. You can update by downloading from https://www.terraform.io/downloads.html
{
    "UserId": "AIDAVGTZUR6UYK7CKXWNS",
    "Account": "357811130281",
    "Arn": "arn:aws:iam::357811130281:user/sean-study"
}


In [2]:
import os
base="D:/Workspace/Technologies/citi_terraform"
os.makedirs(base,exist_ok=True)
print("Created:",base)

Created: D:/Workspace/Technologies/citi_terraform


In [3]:
import os

base = "D:/Workspace/Technologies/citi_terraform"
os.makedirs(base, exist_ok=True)

files = {
    "versions.tf": """\
terraform {
  required_version = ">=1.5"
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~>5.0"
    }
  }
}

provider "aws" {
  region = var.aws_region
}
""",
    "variables.tf": """\
variable "aws_region" {
  default = "us-east-1"
}

variable "project_name" {
  default = "citi-telemetry"
}

variable "environment" {
  default = "dev"
}
""",
    "main.tf": """\
data "aws_caller_identity" "current" {}

resource "aws_s3_bucket" "data_lake" {
  bucket = "${var.project_name}-data-lake-${var.environment}-${data.aws_caller_identity.current.account_id}"
}
""",
    "outputs.tf": """\
output "bucket_name" {
  value = aws_s3_bucket.data_lake.bucket
}
""",
}

for filename, content in files.items():
    with open(os.path.join(base, filename), "w", encoding="utf-8") as f:
        f.write(content)

print("HCL files written to", base)
for f in files:
    print(f"  {f}")

HCL files written to D:/Workspace/Technologies/citi_terraform
  versions.tf
  variables.tf
  main.tf
  outputs.tf


In [4]:
cwd = "D:/Workspace/Technologies/citi_terraform"
result = subprocess.run(["terraform", "init"], cwd=cwd, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(result.stdout if result.stdout else result.stderr)

Initializing the backend...
Initializing provider plugins...
- Finding hashicorp/aws versions matching "~> 5.0"...
- Installing hashicorp/aws v5.100.0...
- Installed hashicorp/aws v5.100.0 (signed by HashiCorp)
Terraform has created a lock file .terraform.lock.hcl to record the provider
selections it made above. Include this file in your version control repository
so that Terraform can guarantee to make the same selections by default when
you run "terraform init" in the future.

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.



In [5]:
result = subprocess.run(["terraform", "plan"], cwd="D:/Workspace/Technologies/citi_terraform", capture_output=True, text=True, encoding="utf-8", errors="replace")
print(result.stdout if result.stdout else result.stderr)

data.aws_caller_identity.current: Reading...
data.aws_caller_identity.current: Read complete after 0s [id=357811130281]

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # aws_s3_bucket.data_lake will be created
  + resource "aws_s3_bucket" "data_lake" {
      + acceleration_status         = (known after apply)
      + acl                         = (known after apply)
      + arn                         = (known after apply)
      + bucket                      = "citi-telemetry-data-lake-dev-357811130281"
      + bucket_domain_name          = (known after apply)
      + bucket_prefix               = (known after apply)
      + bucket_regional_domain_name = (known after apply)
      + force_destroy               = false
      + hosted_zone_id              = (known after apply)
      + id                          = (known after apply)
    

In [6]:
result = subprocess.run(["terraform", "apply", "-auto-approve"], cwd="D:/Workspace/Technologies/citi_terraform", capture_output=True, text=True, encoding="utf-8", errors="replace")
print(result.stdout if result.stdout else result.stderr)

data.aws_caller_identity.current: Reading...
data.aws_caller_identity.current: Read complete after 0s [id=357811130281]

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # aws_s3_bucket.data_lake will be created
  + resource "aws_s3_bucket" "data_lake" {
      + acceleration_status         = (known after apply)
      + acl                         = (known after apply)
      + arn                         = (known after apply)
      + bucket                      = "citi-telemetry-data-lake-dev-357811130281"
      + bucket_domain_name          = (known after apply)
      + bucket_prefix               = (known after apply)
      + bucket_regional_domain_name = (known after apply)
      + force_destroy               = false
      + hosted_zone_id              = (known after apply)
      + id                          = (known after apply)
    

## Terraform State
Tracks mapping between config and real infra. Never edit manually.

In [7]:
result = subprocess.run(["terraform", "state", "list"], cwd="D:/Workspace/Technologies/citi_terraform", capture_output=True, text=True, encoding="utf-8", errors="replace")
print(result.stdout if result.stdout else result.stderr)

data.aws_caller_identity.current
aws_s3_bucket.data_lake



In [8]:
result = subprocess.run(["aws", "s3", "ls"], capture_output=True, text=True, encoding="utf-8", errors="replace")
print(result.stdout if result.stdout else result.stderr)

2026-01-27 19:16:00 aws-capacity-forecaster-sean
2026-03-31 22:41:44 citi-telemetry-data-lake-dev-357811130281
2026-01-10 16:15:48 egirg-cfn-site-1768083341
2026-01-12 23:07:12 egirgis-datalake-v1
2026-01-10 16:38:35 egirgis-lab
2026-01-13 00:05:45 horizonscale-datalake-v1
2026-01-28 11:09:06 sagemaker-us-east-1-357811130281
2026-01-21 20:02:20 sean-capacity-forecast-data



In [9]:
result = subprocess.run(["terraform", "destroy", "-auto-approve"], cwd="D:/Workspace/Technologies/citi_terraform", capture_output=True, text=True, encoding="utf-8", errors="replace")
print(result.stdout if result.stdout else result.stderr)

data.aws_caller_identity.current: Reading...
data.aws_caller_identity.current: Read complete after 0s [id=357811130281]
aws_s3_bucket.data_lake: Refreshing state... [id=citi-telemetry-data-lake-dev-357811130281]

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  - destroy

Terraform will perform the following actions:

  # aws_s3_bucket.data_lake will be destroyed
  - resource "aws_s3_bucket" "data_lake" {
      - arn                         = "arn:aws:s3:::citi-telemetry-data-lake-dev-357811130281" -> null
      - bucket                      = "citi-telemetry-data-lake-dev-357811130281" -> null
      - bucket_domain_name          = "citi-telemetry-data-lake-dev-357811130281.s3.amazonaws.com" -> null
      - bucket_regional_domain_name = "citi-telemetry-data-lake-dev-357811130281.s3.us-east-1.amazonaws.com" -> null
      - force_destroy               = false -> null
      - hosted_zone_id        

## Summary
IaC → plan → apply → state → destroy